# JSON schema mode
Native structured output: the response is constrained to a JSON schema generated from a Pydantic model.

In [ ]:
import json
import time

from dotenv import load_dotenv
from google import genai
from google.genai import types
from IPython.display import JSON, Markdown, display

load_dotenv()  # loads GEMINI_API_KEY from .env
client = genai.Client()

from demo_helpers import VALID_APPLICATIONS

In [ ]:
transcript = "Hey Google, navigate to Ara."

In [ ]:
from pydantic import BaseModel, Field


class ParsedCommand(BaseModel):
    target_application: str = Field(description="The application the user wants to use")
    target_action: str = Field(
        description="The action to perform within that application"
    )
    additional_parameters: dict[str, str] = Field(
        default_factory=dict,
        description=(
            "Parameters for the action mentioned in the transcript, as a mapping "
            "of parameter name (as listed for the chosen action) to value"
        ),
    )


JSON(ParsedCommand.model_json_schema())

## Input

In [ ]:
message = f"""For the following transcript, determine the target action the user wants to take,
the target application, and any additional parameters.\n
The target application, action and parameters must be chosen from the following
valid applications (each lists its actions and the parameters each action accepts):\n
{json.dumps(VALID_APPLICATIONS, indent=2)}\n

transcript: {transcript}"""
print(message)

In [ ]:
config = types.GenerateContentConfig(
    response_mime_type="application/json",
    response_json_schema=ParsedCommand.model_json_schema(),
)
config

## Model call

In [ ]:
start = time.perf_counter()
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=message,
    config=config,
)
print(f"Client call took {time.perf_counter() - start:.2f}s")

## Output

In [ ]:
response

In [ ]:
# response.parsed is the already-decoded JSON (a dict); validate it into the Pydantic model
result = ParsedCommand.model_validate(response.parsed)
result